In [4]:
# ▶ Warnings 제거
import warnings
warnings.filterwarnings('ignore')

# ▶ Google drive mount or 폴더 클릭 후 구글드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

# ▶ 경로 설정 (※ Colab을 활성화시켰다면 보통 Colab Notebooks 폴더가 자동 생성)
import os
os.chdir("/content/drive/MyDrive/Colab python")
os.getcwd()

import pandas as pd
df = pd.read_csv('top_insta_influencers_data.csv')
df.head()

Mounted at /content/drive


,rank,channel_info,influence_score,posts,followers,avg_likes,60_day_eng_rate,new_post_avg_like,total_likes,country
0,1,cristiano,92,3.3k,475.8m,8.7m,1.39%,6.5m,29.0b,Spain
1,2,kyliejenner,91,6.9k,366.2m,8.3m,1.62%,5.9m,57.4b,United States
2,3,leomessi,90,0.89k,357.3m,6.8m,1.24%,4.4m,6.0b,NaN
3,4,selenagomez,93,1.8k,342.7m,6.2m,0.97%,3.3m,11.5b,United States
4,5,therock,91,6.8k,334.1m,1.9m,0.20%,665.3k,12.5b,United States


In [5]:
print("데이터 Shape:", df.shape)

데이터 Shape: (200, 10)


In [6]:
print(df.dtypes)

rank                  int64
channel_info         object
influence_score       int64
posts                object
followers            object
avg_likes            object
60_day_eng_rate      object
new_post_avg_like    object
total_likes          object
country              object
dtype: object


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   rank               200 non-null    int64 
 1   channel_info       200 non-null    object
 2   influence_score    200 non-null    int64 
 3   posts              200 non-null    object
 4   followers          200 non-null    object
 5   avg_likes          200 non-null    object
 6   60_day_eng_rate    200 non-null    object
 7   new_post_avg_like  200 non-null    object
 8   total_likes        200 non-null    object
 9   country            138 non-null    object
dtypes: int64(2), object(8)
memory usage: 15.8+ KB


In [8]:
df.isnull()

,rank,channel_info,influence_score,posts,followers,avg_likes,60_day_eng_rate,new_post_avg_like,total_likes,country
0,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,True
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
195,False,False,False,False,False,False,False,False,False,False
196,False,False,False,False,False,False,False,False,False,False
197,False,False,False,False,False,False,False,False,False,False
198,False,False,False,False,False,False,False,False,False,False


In [9]:
print(df.isnull().sum())

rank                  0
channel_info          0
influence_score       0
posts                 0
followers             0
avg_likes             0
60_day_eng_rate       0
new_post_avg_like     0
total_likes           0
country              62
dtype: int64


In [10]:
df.describe()

,rank,influence_score
count,200.000000,200.000000
mean,100.500000,81.820000
std,57.879185,8.878159
min,1.000000,22.000000
25%,50.750000,80.000000
50%,100.500000,84.000000
75%,150.250000,86.000000
max,200.000000,93.000000


In [11]:
df['country'] = df['country'].fillna('Unknown')
print(df.isnull().sum())
# 인플루언서 영향력을 판단하는데 중요하지 않은 country 컬럼의 null 값은 'Unknown'으로 대체

rank                 0
channel_info         0
influence_score      0
posts                0
followers            0
avg_likes            0
60_day_eng_rate      0
new_post_avg_like    0
total_likes          0
country              0
dtype: int64


In [12]:
def convert_str_to_number(value):
    """
    문자열(k, m, b, %) 형태의 데이터를 숫자형(float)으로 변환하는 함수
    """
    if pd.isna(value):
        return 0.0

    value = str(value).lower().strip()

    # % 제거
    if '%' in value:
        return float(value.replace('%', '')) / 100

    # 단위 기호 처리
    multiplier = 1
    if 'k' in value:
        multiplier = 1_000
        value = value.replace('k', '')
    elif 'm' in value:
        multiplier = 1_000_000
        value = value.replace('m', '')
    elif 'b' in value:
        multiplier = 1_000_000_000
        value = value.replace('b', '')

    try:
        return float(value) * multiplier
    except ValueError:
        return 0.0

# 3. 변환이 필요한 컬럼 리스트
target_cols = ['posts', 'followers', 'avg_likes', '60_day_eng_rate', 'new_post_avg_like', 'total_likes']

# 4. 데이터 변환 적용
for col in target_cols:
    df[col] = df[col].apply(convert_str_to_number)

# --- [테스트 코드] ---
print("1. 결측치 처리 확인 (country):", df['country'].isnull().sum(), "개 남음")
print("2. 데이터 타입 확인:\n", df[target_cols].dtypes)
print("\n3. 변환 결과 샘플 (상위 5행):")
print(df[['channel_info', 'followers', 'total_likes', '60_day_eng_rate']].head())

1. 결측치 처리 확인 (country): 0 개 남음
2. 데이터 타입 확인:
 posts                float64
followers            float64
avg_likes            float64
60_day_eng_rate      float64
new_post_avg_like    float64
total_likes          float64
dtype: object

3. 변환 결과 샘플 (상위 5행):
  channel_info    followers   total_likes  60_day_eng_rate
0    cristiano  475800000.0  2.900000e+10           0.0139
1  kyliejenner  366200000.0  5.740000e+10           0.0162
2     leomessi  357300000.0  6.000000e+09           0.0124
3  selenagomez  342700000.0  1.150000e+10           0.0097
4      therock  334100000.0  1.250000e+10           0.0020


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rank               200 non-null    int64  
 1   channel_info       200 non-null    object 
 2   influence_score    200 non-null    int64  
 3   posts              200 non-null    float64
 4   followers          200 non-null    float64
 5   avg_likes          200 non-null    float64
 6   60_day_eng_rate    199 non-null    float64
 7   new_post_avg_like  200 non-null    float64
 8   total_likes        200 non-null    float64
 9   country            200 non-null    object 
dtypes: float64(6), int64(2), object(2)
memory usage: 15.8+ KB


In [14]:
# 영향력 판단을 위한 가설 3가지
# 1.소통의 밀도 가설: 팔로워 수가 절대적으로 많지 않더라도, 게시물당 참여율(60_day_eng_rate)이 높은 인플루언서가 실질적인 팬덤 영향력은 더 클 것이다.
# 2.성장 잠재력 가설: 현재의 influence_score가 낮더라도 최근 게시물 평균 좋아요(new_post_avg_like)가 전체 평균 좋아요(avg_likes)보다 월등히 높은 인플루언서는 급상승 중인 '라이징 스타'일 것이다.
# 3.규모와 최근 트렌드를 더한 값이 크면 영향력이 클 것이다.
# 가설 선택 이유: '현재의 활성도'에 초점_현재의 영향력은 과거의 명성이나 단순한 팬심보다 게시물에 대한 즉각적 반응이 더 중요할 것.


In [15]:
# [가설 검증을 위한 지표 정의 및 계산]
# 1.소통의 밀도: 팔로워 수 대비 참여율이 중요하다. -- 60_day_eng_rate (높을수록 소통 밀도가 높다)
# 2.성장 잠재력: '평소 실력'보다 '최근 실력'이 월등히 높은 사람을 찾아야 합니다. -- avg_likes(과거/평균)과 new_post_avg_like(최근 1개월)의 차이
# 3.지표 가중 (규모 + 트렌드): followers(규모) 50% + new_post_avg_like(트렌드) 50%



In [16]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 6))
sns.scatterplot(data=df, x='followers', y='influence_score', alpha=0.6)

plt.title("Followers vs Influence Score")
plt.xlabel("Followers")
plt.ylabel("Influence Score")
plt.grid(True)
plt.show()

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6600/4054253554.py", line 2, in <cell line: 0>
    import seaborn as sns
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtrac